# Python. Подготовка к семинару 3 — recap и углубление

Личный конспект к семинару «Множества и словари вблизи». Сначала — быстрый
повтор лекции 3 (множества, хешируемость, `array`, словари), затем каждый сюжет
семинара — глубже: дополнительные примеры, трюки и детали CPython. В конце —
викторина из десяти «предскажи, что выведет».

**Как пользоваться**

1. Прогнать все ячейки сверху вниз (Kernel → Restart & Run All).
2. Ячейки самодостаточны: переменные определяются там же, где используются.
3. Викторина: сначала ответь сам — в голове или на бумаге, — потом запускай
   ячейку; разбор лежит под ней.


---

## Часть 1. Быстрый recap лекции

### 1.1. Множества

Множество — неупорядоченный набор уникальных хешируемых элементов. Внутри —
хеш-таблица, поэтому проверка `in` — это вычисление слота по хешу, а не перебор.


In [1]:
a = {3, 1, 2}                        # литерал
b = set("абракадабра")               # из любого итерируемого, повторы схлопываются
c = {x * x for x in range(4) if x}   # set comprehension
empty = set()                        # пустое — только так: {} — это словарь

a, b, c, empty, bool(empty)


({1, 2, 3}, {'д', 'б', 'р', 'к', 'а'}, {1, 4, 9}, set(), False)

Операторы и методы (порядок аргументов у разности важен):


In [2]:
a, b = {1, 2, 3}, {3, 4}
print("объединение :", a | b)
print("пересечение :", a & b)
print("разность    :", a - b, "<->", b - a)
print("симм. разн. :", a ^ b)
print("под/надмн-во:", {1, 2} <= a, a >= {1, 2, 3})
print("без общих   :", {1, 9}.isdisjoint(b))


объединение : {1, 2, 3, 4}
пересечение : {3}
разность    : {1, 2} <-> {4}
симм. разн. : {1, 2, 4}
под/надмн-во: True True
без общих   : True


Один операнд: `add`, `discard` (тихо), `remove` (ругнётся), `pop` (случайный
элемент), `clear`, `copy`.

`frozenset` — неизменяемое множество: те же операции чтения, без мутирующих,
зато хешируемое — годится в ключи и в другие множества.


In [3]:
fs = frozenset({1, 2, 3})
print(fs | {4})            # операции работают, результат — frozenset
print(fs <= {1, 2, 3, 4})


frozenset({1, 2, 3, 4})
True


### 1.2. Хешируемость

Объект хешируем, если у него есть `__hash__` и выполняется контракт: равные
объекты дают равный хеш, хеш не меняется всю жизнь объекта. Неизменяемые
встроенные типы хешируемы; изменяемые (`list`, `set`, `dict`) — нет. Кортеж
хешируем ровно тогда, когда хешируемо его содержимое.


In [4]:
print(hash("строка") == hash("строка"), hash((1, 2)), hash(1.5))
print(hash(-1))          # любопытство CPython: -2

try:
    hash([1, 2])         # изменяемый — не хешируем (исключения — тема лекции 7)
except TypeError as e:
    print("TypeError:", e)

try:
    hash((1, [2]))       # кортеж с изменяемым внутри — тоже нет
except TypeError as e:
    print("TypeError:", e)


True -3550055125485641917 1152921504606846977
-2
TypeError: unhashable type: 'list'
TypeError: unhashable type: 'list'


### 1.3. `array`

`array.array` — как список, но хранит однотипные C-значения: тип задаётся
typecode (`"i"` — int, `"d"` — double), память компактнее, обмен — бинарный.
Индексация и срезы — те же.


In [5]:
from array import array

nums = array("i", range(1000))
py_list = list(range(1000))
print("array :", nums.itemsize, "байт на элемент, итого", nums.itemsize * len(nums))
print("list  :", py_list.__sizeof__(), "байт — и это только указатели,",
      "сами объекты int оплачиваются отдельно")
print(nums[0], nums[-1], len(nums))


array : 4 байт на элемент, итого 4000
list  : 8040 байт — и это только указатели, сами объекты int оплачиваются отдельно
0 999 1000


### 1.4. Словари

Четыре способа создания одного и того же словаря:


In [6]:
literal = {"a": 1, "b": 2}
from_pairs = dict([("a", 1), ("b", 2)])
from_zip = dict(zip("ab", [1, 2]))
comp = {k: v for k, v in zip("ab", [1, 2])}

literal, from_pairs == literal, from_zip == literal, comp == literal


({'a': 1, 'b': 2}, True, True, True)

Повседневные операции:


In [7]:
d = {"хлеб": 50, "молоко": 80}
print(d.get("кефир"), d.get("кефир", 0))   # None, 0
print(d.setdefault("кефир", 0))            # вставил и вернул значение
d.update({"хлеб": 60, "сыр": 300})
print(d.pop("молоко"), d.popitem())        # значение и последняя пара
print("хлеб" in d, list(d))


None 0
0
80 ('сыр', 300)
True ['хлеб', 'кефир']


Слияния и «живые» представления:


In [8]:
prices = {"хлеб": 50}
more = {"молоко": 80, "хлеб": 60}
print({**prices, **more})     # распаковка: при конфликте побеждает правый
print(prices | more)          # оператор слияния (Python 3.9+)

keys = prices.keys()          # представление, а не копия
prices["сыр"] = 300           # изменили словарь после создания keys
print(list(keys), list(prices.items()))


{'хлеб': 60, 'молоко': 80}
{'хлеб': 60, 'молоко': 80}
['хлеб', 'сыр'] [('хлеб', 50), ('сыр', 300)]


---

## Часть 2. Углубление

### 2.1. Ключи: hash + равенство, числовая башня

Словарь и множество считают ключи одинаковыми, если совпал хеш и `==`. Числа в
Python образуют «числовую башню»: `1`, `1.0`, `True`, `Decimal("1")` и
`Fraction(1, 1)` равны друг другу — значит, обязаны иметь одинаковый хеш:


In [9]:
from decimal import Decimal
from fractions import Fraction

hash(1) == hash(1.0) == hash(True) == hash(Decimal("1")) == hash(Fraction(1, 1))


True

Следствия: читать можно по любому «равному» написанию; при записи ключ
остаётся от **первой** вставки, значение — от последней.


In [10]:
d = {1: "целое"}
d[True] = "логическое"     # не новый ключ, а перезапись
d[1.0] = "вещественное"

print(d, type(next(iter(d))).__name__)   # ключ — всё ещё int 1
print(d[True], d[1])                     # читаем любым равным


{1: 'вещественное'} int
вещественное вещественное


Нули: `0.0 == -0.0` — истина, и хеши равны. Это один ключ.


In [11]:
zeros = {0.0, -0.0, 0, False}  # noqa: B033
print(len(zeros), type(next(iter(zeros))).__name__)


1 float


Специальные значения `float`:


In [12]:
nan = float("nan")
print(hash(float("inf")))   # фиксированная константа: 314159
print(hash(-1.0))           # -2: хеш float сводится к хешу равного int
print(hash(nan))            # у каждого nan свой хеш (см. раздел 2.4)


314159
-2
8764150621925


До Python 3.10 у всех `nan` был хеш `0`; теперь хеш привязан к конкретному
объекту — но равенства это не добавило, см. 2.4.

Вывод: не смешивай числа разных типов в ключах. Якорь: `keys.py` с семинара.

### 2.2. Хеш целых: модуль 2⁶¹−1

Хеш целого в CPython — само число по модулю простого Мерсенна `P = 2⁶¹ − 1`.
Малые числа — хеш сам на себя; у больших начинается «наматывание»:


In [13]:
P = 2**61 - 1
print(hash(1), hash(2), hash(3))           # маленькие — сами себя
print(hash(P), hash(P + 1), hash(P + 2))   # 0, 1, 2 — модуль!
print(hash(2 * P), hash(3 * P))            # тоже 0


1 2 3
0 1 2
0 0


Кстати, «отсортированный» вид `{3, 1, 2}` → `{1, 2, 3}` — следствие того же:
слот в таблице считается от хеша, а у малых чисел хеш — само число. Совпадение,
а не гарантия (см. 2.3). А `hash(-1) == -2` — деталь реализации: хеш приводится
к симметричному диапазону, чтобы `-1` не занимал «особое» значение.

Одинаковый хеш — ещё не одинаковый ключ: хеш лишь «адрес», дальше контейнер
сверяет `==`. Коллизия хешей при разных значениях — нормальная жизнь таблицы:


In [14]:
print(hash(0) == hash(P) == hash(2 * P))
print(len({0, P, 2 * P}))   # три разных ключа с одним хешем


True
3


Якорь: ДЗ `hash_table` — там коллизии и цепочки собираются руками.

### 2.3. Рандомизация хеша строк: порядок множества не наш

Хеш `str` и `bytes` зависит от случайного числа, которое интерпретатор выбирает
при старте, — защита от подбора «плохих» ключей, превращающих словарь в
связный список. Поэтому порядок обхода множества строк меняется от запуска
к запуску — три независимых интерпретатора:


In [15]:
import subprocess
import sys

for _ in range(3):
    out = subprocess.run(
        [sys.executable, "-c", "print({'а', 'б', 'в', 'г', 'д'})"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    print(out)


{'в', 'б', 'а', 'д', 'г'}
{'а', 'д', 'г', 'б', 'в'}
{'а', 'б', 'в', 'г', 'д'}


Зафиксировать можно переменной окружения `PYTHONHASHSEED=0` — рандомизация
отключается, порядок становится воспроизводимым (так делают в тестах):


In [16]:
import os
import subprocess
import sys

for _ in range(3):
    out = subprocess.run(
        [sys.executable, "-c", "print({'а', 'б', 'в', 'г', 'д'})"],
        capture_output=True,
        text=True,
        check=True,
        env={**os.environ, "PYTHONHASHSEED": "0"},
    ).stdout.strip()
    print(out)


{'д', 'б', 'в', 'г', 'а'}
{'д', 'б', 'в', 'г', 'а'}
{'д', 'б', 'в', 'г', 'а'}


Вывод: единственный порядок, на который можно положиться, — заданный явно через
`sorted`. «Первый элемент множества» не существует. Якорь: `sets.py` — поэтому
результаты там возвращаются отсортированным списком.

### 2.4. `nan`: идентичность выручает, но не спасает

`nan` не равен ничему, включая себя. Поиск в множестве: найти слот по хешу,
сравнить `==`. Но перед `==` контейнер проверяет, не тот ли это самый объект
(`is`) — оптимизация, которая здесь меняет всё:


In [17]:
nan = float("nan")
print(nan == nan, nan in {nan})                        # False, True: тот же объект
print(len({nan, nan}), len({float("nan"), float("nan")}))


False True
1 2


В словаре та же механика: «свой» `nan` достаётся, чужой становится новой
записью:


In [18]:
nan = float("nan")
d = {nan: "первый"}
print(d[nan])                  # нашёлся: тот же объект
d[float("nan")] = "второй"     # не равен и не тот же — новая запись
print(len(d), list(d.values()))


первый
2 ['первый', 'второй']


Вывод: `nan` не годится ни в ключи, ни в элементы множества. Проверять на него
нужно `math.isnan` — сравнение не работает в принципе.

### 2.5. Оператор строже метода

`a | b` требует, чтобы справа было множество; метод принимает любую
последовательность:


In [19]:
a, b = {1, 2}, [3, 4]
print(a.union(b))            # метод: любой итерируемый справа

try:
    a | b                    # оператор: справа должно быть множество
except TypeError as e:
    print("TypeError:", e)


{1, 2, 3, 4}
TypeError: unsupported operand type(s) for |: 'set' and 'list'


Строка — тоже последовательность, и метод молча разберёт её посимвольно.
Классическая ловушка, когда «множество символов» склеивают со словом:


In [20]:
print({"а", "б"}.union("вг"))


{'в', 'а', 'б', 'г'}


Тип результата: у оператора побеждает тип левого операнда, у метода — тип того,
у кого вызвали:


In [21]:
print(type(frozenset({1}) | {2}).__name__)
print(type({1}.union(frozenset({2}))).__name__)


frozenset
set


`|=` у множества меняет его на месте; у `frozenset` операций на месте нет —
он неизменяемый:


In [22]:
s = {1}
s |= {2}
print(s)

try:
    fs = frozenset(s)
    fs |= {3}
except TypeError as e:
    print("TypeError:", e)


{1, 2}


### 2.6. `get` против `in` против сторожа

`get` отвечает `None` и на «ключа нет», и на «значение `None`». Различает их
`in`; альтернатива — сторожевой объект:


In [23]:
settings = {"прокси": None}          # «прокси не нужен» — настройка задана!
print(settings.get("прокси"), settings.get("шрифт"))
print("прокси" in settings, "шрифт" in settings)

MISSING = object()                   # равен только самому себе
print(settings.get("прокси", MISSING) is MISSING)   # False: ключ есть
print(settings.get("шрифт", MISSING) is MISSING)    # True: ключа нет


None None
True False
False
True


Родственная грабля — писать `if d.get(k):` вместо `if k in d:`: ноль и пустая
строка «съедаются» булевым контекстом:


In [24]:
settings = {"таймаут": 0}
if settings.get("таймаут"):
    print("таймаут задан")
else:
    print("таймаут не задан")     # напечатает это: 0 ложен


таймаут не задан


Якорь: `lookup.py` — весь построен на этом различии.

### 2.7. `setdefault`: словарь списков за один проход


In [25]:
words = ["кот", "пёс", "як", "коза"]

groups: dict[int, list[str]] = {}
for word in words:
    if len(word) not in groups:
        groups[len(word)] = []
    groups[len(word)].append(word)

same: dict[int, list[str]] = {}
for word in words:
    same.setdefault(len(word), []).append(word)

print(groups, groups == same)


{3: ['кот', 'пёс'], 2: ['як'], 4: ['коза']} True


Цена: пустой список создаётся на **каждой** итерации и выбрасывается, если ключ
уже есть. Для `[]` это копейки; дорогой дефолт так писать не стоит.

Для счётчика есть брат-паттерн через `get` — без списков вообще:


In [26]:
text = "абракадабра"
counts: dict[str, int] = {}
for ch in text:
    counts[ch] = counts.get(ch, 0) + 1
print(counts)


{'а': 5, 'б': 2, 'р': 2, 'к': 1, 'д': 1}


Включением группировку не решить: comprehension строит значение для каждого
ключа независимо и не умеет накапливать. `defaultdict` и `Counter` — лекция 10.

Якорь: `grouping.py`; ДЗ `group_by_length`, `char_counts`, `word_frequency`.

### 2.8. Порядок словаря: что гарантировано

Гарантия (с Python 3.7): ключи хранятся в порядке вставки. Перезапись значения
не двигает ключ; удаление и новая вставка — двигает в конец; `popitem()` снимает
с конца:


In [27]:
d = {"a": 1, "b": 2}
d["a"] = 9            # перезапись: ключ остаётся на месте
print(list(d))
del d["a"]
d["a"] = 0            # свежая вставка: в конец
print(list(d))
print(d.popitem())    # LIFO: последний вставленный, а не последний тронутый
print(d)


['a', 'b']
['b', 'a']
('a', 0)
{'b': 2}


`==` порядок игнорирует — словари равны по содержимому. Слияния `{**a, **b}` и
`a | b`: ключи `a`, затем новые из `b`; конфликт значений решает правый:


In [28]:
a = {"a": 1, "b": 2}
b = {"b": 20, "c": 3}
merged = {**a, **b}
print(merged, list(merged))
print({"b": 2, "a": 1} == a)            # порядок не важен
print(dict(sorted(merged.items())))     # сортировка по ключу


{'a': 1, 'b': 20, 'c': 3} ['a', 'b', 'c']
True
{'a': 1, 'b': 20, 'c': 3}


Якорь: ДЗ `merge_prices` (слить, не мутируя входы), `invert_dict` («побеждает
поздний» — это то же правило перезаписи).

### 2.9. Представления: живые и множественные

`keys()`, `values()`, `items()` — не копии, а окна в словарь: изменился словарь —
изменился вид. Поддерживают `len`, `in`, итерацию; `keys` и `items` ведут себя
как множества:


In [29]:
d = {"a": 1, "b": 2}
keys = d.keys()
d["c"] = 3                    # после создания представления!
print(list(keys), len(keys), "c" in keys)

other = {"b": 2, "c": 30}
print(d.keys() & other.keys())       # общие ключи
print(d.items() & other.items())     # общие пары — сравнение целиком

try:
    d.values() & other.values()      # values — не множество
except TypeError as e:
    print("TypeError:", e)


['a', 'b', 'c'] 3 True
{'b', 'c'}
{('b', 2)}
TypeError: unsupported operand type(s) for &: 'dict_values' and 'dict_values'


Менять словарь во время итерации по нему нельзя — интерпретатор следит за
размером. Правильный обход со «снимком» ключей:


In [30]:
d = {"a": 1, "b": 2, "c": 3}
try:
    for key in d:
        d[key + "!"] = d[key]
except RuntimeError as e:
    print("RuntimeError:", e)
print("успело добавиться:", len(d) - 3)

d = {"a": 1, "b": 2, "c": 3}
for key in list(d):           # снимок ключей — менять можно
    d[key + "!"] = d[key]
print(sorted(d))


RuntimeError: dictionary changed size during iteration
успело добавиться: 1
['a', 'a!', 'b', 'b!', 'c', 'c!']


Якорь: `lookup.py` — `missing` естественно пишется через разность представлений.

### 2.10. `frozenset`: хешируемое множество

Неизменяемость → хеш → можно ключом и элементом множества. Задача «сгруппировать
по составу букв»: ключ — `frozenset`. Помни: он теряет кратность букв; строгим
анаграммам нужен отсортированный кортеж:


In [31]:
words = ["па", "папа", "ап"]

by_letters: dict[frozenset[str], list[str]] = {}
strict: dict[tuple[str, ...], list[str]] = {}
for word in words:
    by_letters.setdefault(frozenset(word), []).append(word)
    strict.setdefault(tuple(sorted(word)), []).append(word)

print("состав букв:", by_letters)
print("строгие     :", strict)


состав букв: {frozenset({'п', 'а'}): ['па', 'папа', 'ап']}
строгие     : {('а', 'п'): ['па', 'ап'], ('а', 'а', 'п', 'п'): ['папа']}


Фишки: хеш `frozenset` считается один раз и кэшируется; `copy()` возвращает
тот же самый объект (неизменяемому копия без надобности), у `set` — новый:


In [32]:
fs, s = frozenset({1, 2}), {1, 2}
print(fs.copy() is fs, s.copy() is s)
print(hash(fs) == hash(fs.copy()))


True False
True


### 2.11. Мешок трюков

Объединение любого числа множеств и дедупликация с сохранением порядка:


In [33]:
many = [{1, 2}, {2, 3}, {5}]
print(set().union(*many))     # объединение многих
print({*many[0], *many[1]})   # …или распаковкой в литерал

print(list(dict.fromkeys(["б", "а", "б", "а"])))   # дедуп с порядком
print(dict.fromkeys(["ширина", "высота"], 0))      # инициализация дефолтом


{1, 2, 3, 5}
{1, 2, 3}
['б', 'а']
{'ширина': 0, 'высота': 0}


Ключ с максимальным значением, первый вставленный ключ, слияние без мутации:


In [34]:
counts = {"а": 3, "б": 7, "в": 1}
print(max(counts, key=counts.get))   # ключ с максимальным значением
print(next(iter(counts)))            # первый вставленный
print(counts | {"г": 9}, len(counts))   # новый словарь; оригинал цел


б
а
{'а': 3, 'б': 7, 'в': 1, 'г': 9} 3


Почему литерал быстрее конструктора — видно в байткоде. Компилятор сворачивает
константы: у литерала — `BUILD_SET` плюс готовая константа-`frozenset` и
`SET_UPDATE`; у конструктора — поиск имени `set`, построение списка и вызов
функции в рантайме:


In [35]:
import dis

print("--- {1, 2, 3} ---")
dis.dis("{1, 2, 3}")
print("--- set([1, 2, 3]) ---")
dis.dis("set([1, 2, 3])")


--- {1, 2, 3} ---
  0           RESUME                   0

  1           BUILD_SET                0
              LOAD_CONST               0 (frozenset({1, 2, 3}))
              SET_UPDATE               1
              RETURN_VALUE
--- set([1, 2, 3]) ---
  0           RESUME                   0

  1           LOAD_NAME                0 (set)
              PUSH_NULL
              BUILD_LIST               0
              LOAD_CONST               0 ((1, 2, 3))
              LIST_EXTEND              1
              CALL                     1
              RETURN_VALUE


Включения тоже с оптимизацией: контейнер создаётся один раз, элементы
добавляются на месте инструкцией `SET_ADD` (у словарного — `MAP_ADD`), без
пересборки:


In [36]:
import dis

dis.dis("{x for x in range(3)}")


   0           RESUME                   0

   1           LOAD_NAME                0 (range)
               PUSH_NULL
               LOAD_CONST               0 (3)
               CALL                     1
               GET_ITER
               LOAD_FAST_AND_CLEAR      0 (x)
               SWAP                     2
       L1:     BUILD_SET                0
               SWAP                     2
               GET_ITER
       L2:     FOR_ITER                 4 (to L3)
               STORE_FAST_LOAD_FAST     0 (x, x)
               SET_ADD                  2
               JUMP_BACKWARD            6 (to L2)
       L3:     END_FOR
               POP_TOP
       L4:     SWAP                     2
               STORE_FAST               0 (x)
               RETURN_VALUE

  --   L5:     SWAP                     2
               POP_TOP

   1           SWAP                     2
               STORE_FAST               0 (x)
               RERAISE                  0
ExceptionTable:
  L1 to 

И главное «зачем всё это»: цена проверки `in`. Ищем элемент в конце списка
из 100 000 чисел (худший случай для списка) против множества:


In [37]:
import timeit

haystack_list = list(range(100_000))
haystack_set = set(haystack_list)
needle = 99_999

t_list = timeit.timeit("needle in haystack_list", globals=globals(), number=100)
t_set = timeit.timeit("needle in haystack_set", globals=globals(), number=100_000)
print(f"список   : {t_list / 100 * 1e6:8.1f} мкс на проверку")
print(f"множество: {t_set / 100_000 * 1e6:8.3f} мкс на проверку")
print(f"разница  : ~{t_list / 100 / (t_set / 100_000):,.0f} раз")


список   :    467.8 мкс на проверку
множество:    0.017 мкс на проверку
разница  : ~27,126 раз


Вывод: membership-проверки на больших данных — множества и словари. Именно
поэтому «уже встречалось?» держат во множестве. Якорь: `orders.py`; ДЗ
`dedup_keeping_order` — там это требование явное.

---

## Часть 3. Викторина «предскажи, что выведет»

Десять вопросов — по одному на раздел углубления. Ответил сам → запустил ячейку
→ сверился с разбором. 10 из 10 без единого запуска — к семинару готов.

### Вопрос 1


In [38]:
s = {1, True, 1.0}  # noqa: B033
print(len(s), type(next(iter(s))).__name__)


1 int


**Разбор.** `1 int`. Все три записи — один ключ: значения равны и хеши равны.
Побеждает первый записанный — `int 1`. В словаре значение взяло бы последнее,
но у множества значений нет.

### Вопрос 2


In [39]:
d = {True: "a", 1: "b"}  # noqa: F601
print(d, type(next(iter(d))).__name__)


{True: 'b'} bool


**Разбор.** `{True: 'b'}` и тип `bool`. Ключ — от первой записи (`True`),
значение — от последней (`"b"`).

### Вопрос 3


In [40]:
P = 2**61 - 1
print(hash(P) == hash(0), len({0, P}))


True 2


**Разбор.** `True 2`. Хеши совпали — модуль Мерсенна, — но значения не равны,
и это два разных ключа. Хеш — «адрес», равенство — отдельная проверка.

### Вопрос 4


In [41]:
try:
    {1, 2} | [3]
except TypeError as e:
    print("TypeError:", e)


TypeError: unsupported operand type(s) for |: 'set' and 'list'


**Разбор.** `TypeError: unsupported operand type(s) for |: 'set' and 'list'`.
Оператор требует множество с обеих сторон; для списка справа — `union([3])`.

### Вопрос 5


In [42]:
print(sorted({"а"}.union("вг")))


['а', 'в', 'г']


**Разбор.** `['а', 'в', 'г']`. Метод принял строку-последовательность и разобрал
её посимвольно; `sorted` — единственный честный порядок для множества.

### Вопрос 6


In [43]:
settings = {"прокси": None}
print(settings.get("прокси"), settings.get("шрифт"))
print("прокси" in settings, "шрифт" in settings)


None None
True False


**Разбор.** `None None`, затем `True False`. `get` не отличает «значение `None`»
от «ключа нет» — различает только `in` (или сторожевой объект).

### Вопрос 7


In [44]:
merged = {**{"a": 1, "b": 2}, **{"b": 3, "c": 4}}
print(merged, list(merged))


{'a': 1, 'b': 3, 'c': 4} ['a', 'b', 'c']


**Разбор.** `{'a': 1, 'b': 3, 'c': 4}` и `['a', 'b', 'c']`. Конфликт `b` решён
правым словарём, но ключ `b` остался на месте — перезапись не двигает. `c`
дописалась в конец.

### Вопрос 8


In [45]:
d = {"a": 1, "b": 2}
d["a"] = 9
print(d.popitem(), list(d))


('b', 2) ['a']


**Разбор.** `('b', 2)` и `['a']`. `popitem` берёт последнюю **вставленную**
запись (`b`), а не последнюю тронутую: перезапись `a` не двигала ключ.

### Вопрос 9


In [46]:
zeros = {0.0, -0.0, 0, False}  # noqa: B033
print(len(zeros), type(next(iter(zeros))).__name__)


1 float


**Разбор.** `1 float`. `0.0 == -0.0`, хеши равны — один ключ; `0` и `False`
свалялись туда же. Остаётся первый записанный: `0.0` типа `float`.

### Вопрос 10


In [47]:
print(list(dict.fromkeys(["б", "а", "б"])))


['б', 'а']


**Разбор.** `['б', 'а']`. `fromkeys` вставляет ключи по порядку появления;
повторная вставка существующего ключа не двигает его и не добавляет новый.
Дедупликация с сохранением порядка — до `dict`-трюка её делали именно так.

---

## Финал. Сюжет → где живёт

| Сюжет | Семинар | ДЗ |
|---|---|---|
| равные ключи, тип выжившего | `keys.py` | `invert_dict` (кто побеждает) |
| операции множеств со списков | `sets.py` | `common_elements`, `unique_sorted` |
| `get`/`in` при значении `None` | `lookup.py` | `top_n` (порядок пар) |
| группировка `setdefault`/`get` | `grouping.py` | `group_by_length`, `char_counts`, `word_frequency` |
| множество как память, скорость | `orders.py` | `dedup_keeping_order` |
| хеш-таблица и коллизии | — | `hash_table` |

Если викторина прошла без запусков — семинар не застанет врасплох ✅
